# Two-threshold directional change

This notebook introduces two independent directional-change clocks on the same EURUSD close series:

$$\theta_s = 0.2\%, \qquad \theta_L = 0.4\%$$

The objective is geometric and causal intuition—not an HMM, forecast, or trading rule. We will inspect completed-event ledgers, live modes, nesting, and small-scale reversals that later propagate or fail.

In [1]:
from enum import StrEnum
from pathlib import Path

import numpy as np
import pandas as pd

from dc_basics.core import dc_live_features, parse_dual_dc
from dc_basics.plotting import (
    plot_dc_counter_scale_price_casebook,
    plot_dc_counter_scale_trajectories,
    plot_two_threshold_directional_change,
    plot_two_threshold_modes,
)

## The object we are studying

Each threshold independently maintains its own mode, running extreme, DCC confirmations, and completed TMV, T, and R. The small clock fires more often and, in practice, confirms a reversal before the large clock does. That ordering is a near-invariant rather than a law: with multiplicative thresholds the large clock can, very rarely, confirm first. A small mode opposite to the large mode is a counter-scale move; it is not yet a large-scale reversal.

A completed up leg ends at a peak and becomes knowable at a later **downward DCC**, which switches the live mode down. A completed down leg ends at a trough and becomes knowable at an **upward DCC**, which switches the live mode up. That semantic rule applies separately at both thresholds.

In [2]:
class Scale(StrEnum):
    SMALL = "small"
    LARGE = "large"


from dc_basics.data import resolve_data_root
from dc_basics.enums import AttemptOutcome, DcMode


DATA_ROOT = resolve_data_root()
SYMBOL = "EURUSD.r"
DATA_PATH = DATA_ROOT / f"{SYMBOL}.parquet"
SMALL_THETA = 0.002
LARGE_THETA = 0.004
N_PRICES = 5_000

if not 0.0 < SMALL_THETA < LARGE_THETA:
    raise ValueError("Thresholds must satisfy 0 < SMALL_THETA < LARGE_THETA.")

print(f"Symbol: {SYMBOL}")
print(f"Small threshold: {SMALL_THETA:.3%}")
print(f"Large threshold: {LARGE_THETA:.3%}")
print(f"Chart window: {N_PRICES:,} bars")

Symbol: EURUSD.r
Small threshold: 0.200%
Large threshold: 0.400%
Chart window: 5,000 bars


## Load and audit the bars

DC parsing is path-dependent. We therefore refuse unsorted, duplicated, non-finite, or non-positive prices rather than silently repairing them.

In [3]:
raw = pd.read_parquet(DATA_PATH)
if not isinstance(raw.index, pd.DatetimeIndex):
    if "time" not in raw.columns:
        raise TypeError("Expected a DatetimeIndex or a 'time' column.")
    raw = raw.set_index(pd.to_datetime(raw.pop("time")))

raw = raw.sort_index()
if raw.index.has_duplicates:
    raise ValueError(f"Found {raw.index.duplicated().sum():,} duplicate timestamps.")
if "close" not in raw.columns:
    raise KeyError("The input data has no 'close' column.")

prices = raw["close"].astype(np.float64).rename(SYMBOL)
if not np.isfinite(prices.to_numpy()).all():
    raise ValueError("Close contains NaN or infinite values.")
if prices.le(0.0).any():
    raise ValueError("DC thresholds require strictly positive prices.")

print(f"Bars: {len(prices):,}")
print(f"Start: {prices.index[0]}")
print(f"End: {prices.index[-1]}")
print(f"Timezone: {prices.index.tz}")
raw.tail()

Bars: 258,848
Start: 2023-05-22 14:00:00+02:00
End: 2026-11-09 13:30:00+02:00
Timezone: pytz.FixedOffset(120)


,open,high,low,close,tick_volume,spread
date,,,,,,
2026-11-09 13:10:00+02:00,1.11924,1.11959,1.11922,1.11949,191.0,6
2026-11-09 13:15:00+02:00,1.11948,1.11984,1.11939,1.11958,268.0,6
2026-11-09 13:20:00+02:00,1.11956,1.11956,1.11899,1.11905,194.0,6
2026-11-09 13:25:00+02:00,1.11906,1.11930,1.11900,1.11911,109.0,6
2026-11-09 13:30:00+02:00,1.11912,1.11933,1.11908,1.11921,134.0,6


## Build the two completed-event ledgers

`parse_dual_dc` runs the same event parser independently at each threshold. Event IDs are scale-specific: `S42` and `L42` do not describe the same market event. Every completed value receives `information_time = dcc_idx`.

In [4]:
from dc_basics.events import prepare_event_ledger


small_raw_events, large_raw_events = parse_dual_dc(prices, stheta=SMALL_THETA, btheta=LARGE_THETA)
small_events = prepare_event_ledger(small_raw_events, "S", Scale.SMALL, SMALL_THETA)
large_events = prepare_event_ledger(large_raw_events, "L", Scale.LARGE, LARGE_THETA)

event_counts = pd.DataFrame({
    "threshold": [SMALL_THETA, LARGE_THETA],
    "completed_events": [len(small_events), len(large_events)],
}, index=pd.Index([Scale.SMALL.value, Scale.LARGE.value], name="scale"))
event_counts.style.format({"threshold": "{:.3%}", "completed_events": "{:,}"})

,threshold,completed_events
scale,,
small,0.200%,"2,419"
large,0.400%,755


### Compare completed-event geometry across scales

TMV is measured in that scale's own threshold multiples, T is bars between extremes, and R is normalized displacement per bar. Do not compare event row $i$ across ledgers: the clocks complete at different timestamps.

In [5]:
all_events = pd.concat([small_events, large_events], ignore_index=True)
scale_summary = all_events.groupby("scale", sort=False).agg(
    event_count=("event_id", "size"),
    tmv_mean=("tmv", "mean"),
    tmv_median=("tmv", "median"),
    t_mean=("t", "mean"),
    t_median=("t", "median"),
    r_mean=("r", "mean"),
    r_median=("r", "median"),
)
scale_summary.style.format({
    "event_count": "{:,}",
    "tmv_mean": "{:.4f}",
    "tmv_median": "{:.4f}",
    "t_mean": "{:.2f}",
    "t_median": "{:.2f}",
    "r_mean": "{:.3e}",
    "r_median": "{:.3e}",
})

,event_count,tmv_mean,tmv_median,t_mean,t_median,r_mean,r_median
scale,,,,,,,
small,"2,419",2.2813,1.8990,106.89,64.00,1.373e-04,6.369e-05
large,755,2.1355,1.7848,342.48,238.00,7.442e-05,3.277e-05


## Construct the causal bar-level modes

`dc_live_features` processes prices from left to right. Its direction is the mode known on each bar: `+1 = up`, `-1 = down`. These modes update at DCC bars; unlike completed TMV/T/R, they do not require a future event.

In [6]:
small_live = dc_live_features(prices, theta=SMALL_THETA)
large_live = dc_live_features(prices, theta=LARGE_THETA)
mode_map = {1.0: DcMode.UP.value, -1.0: DcMode.DOWN.value}

mode_frame = pd.DataFrame(index=prices.index)
for scale, live in ((Scale.SMALL, small_live), (Scale.LARGE, large_live)):
    prefix = scale.value
    mode_frame[f"{prefix}_mode"] = live["direction"].map(mode_map)
    mode_frame[f"{prefix}_upward_dcc"] = live["upward_dcc"]
    mode_frame[f"{prefix}_downward_dcc"] = live["downward_dcc"]

small_ids_by_dcc = small_events.set_index("dcc_idx")["event_id"]
large_ids_by_dcc = large_events.set_index("dcc_idx")["event_id"]
mode_frame["small_event_id"] = mode_frame.index.to_series().map(small_ids_by_dcc)
mode_frame["large_event_id"] = mode_frame.index.to_series().map(large_ids_by_dcc)
valid_modes = mode_frame[["small_mode", "large_mode"]].notna().all(axis=1)
mode_frame["mode_pair"] = pd.NA
mode_frame.loc[valid_modes, "mode_pair"] = (
    mode_frame.loc[valid_modes, "small_mode"] + " | " + mode_frame.loc[valid_modes, "large_mode"]
)
mode_frame["modes_agree"] = pd.Series(pd.NA, index=mode_frame.index, dtype="boolean")
mode_frame.loc[valid_modes, "modes_agree"] = (
    mode_frame.loc[valid_modes, "small_mode"] == mode_frame.loc[valid_modes, "large_mode"]
)
mode_frame.tail()

,small_mode,small_upward_dcc,small_downward_dcc,large_mode,large_upward_dcc,large_downward_dcc,small_event_id,large_event_id,mode_pair,modes_agree
date,,,,,,,,,,
2026-11-09 13:10:00+02:00,up,False,False,up,False,False,NaN,NaN,up | up,True
2026-11-09 13:15:00+02:00,up,False,False,up,False,False,NaN,NaN,up | up,True
2026-11-09 13:20:00+02:00,up,False,False,up,False,False,NaN,NaN,up | up,True
2026-11-09 13:25:00+02:00,up,False,False,up,False,False,NaN,NaN,up | up,True
2026-11-09 13:30:00+02:00,up,False,False,up,False,False,NaN,NaN,up | up,True


### Audit the event/live semantic contract

At each scale, completed up-event timestamps must exactly equal live downward-DCC timestamps; completed down-event timestamps must exactly equal live upward-DCC timestamps. This catches the most damaging possible interpretation error before hierarchy analysis.

In [7]:
def assert_event_dcc_alignment(events: pd.DataFrame, live: pd.DataFrame, scale: Scale) -> pd.DataFrame:
    checks = []
    mappings = ((DcMode.UP, "downward_dcc"), (DcMode.DOWN, "upward_dcc"))
    for completed_direction, dcc_flag in mappings:
        event_times = pd.Index(events.loc[events["direction"].eq(completed_direction.value), "dcc_idx"])
        live_times = live.index[live[dcc_flag]]
        missing_from_live = event_times.difference(live_times)
        missing_from_events = live_times.difference(event_times)
        if len(missing_from_live) or len(missing_from_events):
            raise AssertionError(
                f"{scale.value} {completed_direction.value} alignment failed: "
                f"missing_from_live={len(missing_from_live)}, missing_from_events={len(missing_from_events)}"
            )
        checks.append({
            "scale": scale.value,
            "completed_leg": completed_direction.value,
            "confirming_dcc": dcc_flag.replace("_dcc", ""),
            "matched_events": len(event_times),
        })
    return pd.DataFrame(checks)


alignment_audit = pd.concat([
    assert_event_dcc_alignment(small_events, small_live, Scale.SMALL),
    assert_event_dcc_alignment(large_events, large_live, Scale.LARGE),
], ignore_index=True)
alignment_audit

,scale,completed_leg,confirming_dcc,matched_events
0,small,up,downward,1209
1,small,down,upward,1210
2,large,up,downward,377
3,large,down,upward,378


## Overlay both completed-event ledgers

Color and line style encode both scale and direction:

- small up: green solid; small down: red solid;
- large up: blue dashed; large down: orange dashed.

Large legs are thicker. Triangles mark the new live direction at confirmation; diamonds are retrospective ending extremes.

In [8]:
event_fig = plot_two_threshold_directional_change(
    prices=prices,
    small_events=small_events,
    large_events=large_events,
    small_theta=SMALL_THETA,
    large_theta=LARGE_THETA,
    n_prices=N_PRICES,
    title=f"{SYMBOL} | small={SMALL_THETA:.3%}, large={LARGE_THETA:.3%}",
)
event_fig.show()

## View the two live modes without completed-leg clutter

The ribbons are causal. When their directions differ, the small clock has moved against the large clock but has not yet changed the large mode.

In [9]:
mode_fig = plot_two_threshold_modes(
    prices=prices,
    mode_frame=mode_frame,
    n_prices=N_PRICES,
    title=f"{SYMBOL} | causal modes at {SMALL_THETA:.3%} and {LARGE_THETA:.3%}",
)
mode_fig.show()

## Audit 1: occupancy of the four mode combinations

This is bar-time occupancy, not an event probability or trading result. Interpret the combinations as:

| Small | Large | Structural reading |
|---|---|---|
| up | up | upward agreement across scales |
| down | up | small pullback inside large up mode |
| down | down | downward agreement across scales |
| up | down | small rebound inside large down mode |

In [10]:
valid_mode_frame = mode_frame.loc[valid_modes].copy()
mode_occupancy = valid_mode_frame.groupby(["small_mode", "large_mode"], sort=False).agg(
    bars=("mode_pair", "size"),
    small_dccs=("small_event_id", "count"),
    large_dccs=("large_event_id", "count"),
)
mode_occupancy["bar_share"] = mode_occupancy["bars"] / mode_occupancy["bars"].sum()
mode_occupancy.style.format({"bars": "{:,}", "small_dccs": "{:,}", "large_dccs": "{:,}", "bar_share": "{:.2%}"})

## Audit 2: how many small DCCs occur during one large-scale mode?

For each completed large event after the first, we count small DCCs in the causal interval `(previous large DCC, current large DCC]`. The current large DCC closes that large-mode interval. This is a timestamp relationship—not an event-ID join.

In [11]:
large_intervals = large_events[["event_id", "direction", "dcc_idx", "dcc_pos"]].copy()
large_intervals["interval_start_time"] = large_intervals["dcc_idx"].shift(1)
large_intervals["interval_start_pos"] = large_intervals["dcc_pos"].shift(1)
large_intervals = large_intervals.dropna(subset=["interval_start_pos"]).copy()
large_intervals["interval_start_pos"] = large_intervals["interval_start_pos"].astype(np.int64)
small_dcc_positions = small_events["dcc_pos"].to_numpy(dtype=np.int64)
left_edges = np.searchsorted(small_dcc_positions, large_intervals["interval_start_pos"], side="right")
right_edges = np.searchsorted(small_dcc_positions, large_intervals["dcc_pos"], side="right")
large_intervals["small_dcc_count"] = right_edges - left_edges
large_intervals["large_mode_just_completed"] = large_intervals["direction"]

small_count_summary = large_intervals.groupby("large_mode_just_completed", sort=False)["small_dcc_count"].describe(
    percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
)
small_count_summary.style.format(precision=2)

,count,mean,std,min,10%,25%,50%,75%,90%,95%,max
large_mode_just_completed,,,,,,,,,,,
up,377.00,3.06,2.62,1.00,1.00,1.00,3.00,5.00,7.00,7.00,23.00
down,377.00,3.33,3.09,1.00,1.00,1.00,3.00,5.00,7.00,9.40,23.00


## Audit 3: disagreement episodes and propagation

A candidate begins when a small DCC switches the small mode against the large mode known immediately before that bar. It resolves in one of four ways:

- `propagated`: a large DCC changes the large mode to the candidate small direction first;
- `rejected`: a small DCC changes the small mode back to the unchanged large direction first;
- `ambiguous_same_bar`: both resolution mechanisms appear on one bar;
- `unresolved_at_sample_end`: the data ends before resolution.

The candidate direction is known at `start_time`. The outcome is **hindsight only** until `information_time = resolution_time`.

In [12]:
from dc_basics.two_scale import build_disagreement_episodes


propagation_episodes = build_disagreement_episodes(mode_frame, prices)
propagation_episodes.tail(10)

,small_event_id,start_time,start_position,start_price,target_small_mode,new_small_mode,prior_large_mode,outcome,resolution_time,information_time,resolution_position,resolution_bars,resolution_price
1576,S2406,2026-11-02 17:30:00+02:00,257455,1.11996,down,down,up,propagated,2026-11-02 21:50:00+02:00,2026-11-02 21:50:00+02:00,257507,52,1.11911
1577,S2407,2026-11-03 09:20:00+02:00,257645,1.12004,up,up,down,rejected,2026-11-03 13:00:00+02:00,2026-11-03 13:00:00+02:00,257689,44,1.11806
1578,S2409,2026-11-03 20:05:00+02:00,257774,1.11986,up,up,down,propagated,2026-11-04 02:20:00+02:00,2026-11-04 02:20:00+02:00,257849,75,1.12219
1579,S2410,2026-11-04 03:50:00+02:00,257867,1.12333,down,down,up,rejected,2026-11-04 06:30:00+02:00,2026-11-04 06:30:00+02:00,257899,32,1.12557
1580,S2412,2026-11-04 06:55:00+02:00,257904,1.12328,down,down,up,rejected,2026-11-04 10:40:00+02:00,2026-11-04 10:40:00+02:00,257949,45,1.12512
1581,S2414,2026-11-04 17:00:00+02:00,258025,1.12442,down,down,up,propagated,2026-11-04 18:15:00+02:00,2026-11-04 18:15:00+02:00,258040,15,1.12218
1582,S2415,2026-11-05 02:15:00+02:00,258136,1.12386,up,up,down,propagated,2026-11-05 05:45:00+02:00,2026-11-05 05:45:00+02:00,258178,42,1.12615
1583,S2416,2026-11-05 09:25:00+02:00,258222,1.12431,down,down,up,propagated,2026-11-05 13:30:00+02:00,2026-11-05 13:30:00+02:00,258271,49,1.12223
1584,S2417,2026-11-05 17:25:00+02:00,258318,1.12290,up,up,down,rejected,2026-11-05 23:05:00+02:00,2026-11-05 23:05:00+02:00,258386,68,1.12153
1585,S2419,2026-11-06 17:10:00+02:00,258603,1.11560,up,up,down,propagated,2026-11-09 10:10:00+02:00,2026-11-09 10:10:00+02:00,258807,204,1.11775


### Propagation outcome summary

This describes the realized sample. It is not a causal predictor: the outcome and resolution duration use future bars relative to the initiating small DCC.

In [13]:
outcome_summary = propagation_episodes.groupby(["target_small_mode", "outcome"], sort=False).agg(
    episodes=("small_event_id", "size"),
    median_resolution_bars=("resolution_bars", "median"),
    mean_resolution_bars=("resolution_bars", "mean"),
)
outcome_summary

episodes  median_resolution_bars  \
target_small_mode outcome                                        
up                rejected         442                    40.5   
                  propagated       378                    28.5   
down              propagated       377                    32.0   
                  rejected         389                    39.0   

                              mean_resolution_bars  
target_small_mode outcome                           
up                rejected               68.205882  
                  propagated             56.867725  
down              propagated             55.503979  
                  rejected               66.953728

In [14]:
resolved_mask = propagation_episodes["outcome"].isin([
    AttemptOutcome.PROPAGATED.value,
    AttemptOutcome.REJECTED.value,
])
resolved_episodes = propagation_episodes.loc[resolved_mask].copy()
propagation_rates = resolved_episodes.groupby("target_small_mode", sort=False)["outcome"].agg(
    resolved_episodes="size",
    propagation_rate=lambda outcomes: outcomes.eq(AttemptOutcome.PROPAGATED.value).mean(),
)
propagation_rates.loc["both directions"] = {
    "resolved_episodes": len(resolved_episodes),
    "propagation_rate": resolved_episodes["outcome"].eq(AttemptOutcome.PROPAGATED.value).mean(),
}
propagation_rates.style.format({"resolved_episodes": "{:,.0f}", "propagation_rate": "{:.2%}"})

,resolved_episodes,propagation_rate
target_small_mode,,
up,820,46.10%
down,766,49.22%
both directions,"1,586",47.60%


## Audit 4: recent propagated and rejected casebook

The casebook selects the most recent resolved example of each outcome. The black dotted line is the initiating small DCC—the only point at which the candidate exists. The purple dotted line is the later resolution and is hindsight from the start. Read every chart from left to right before looking at the outcome.

In [15]:
casebook = (
    resolved_episodes.sort_values("start_time")
    .groupby("outcome", sort=False, group_keys=False)
    .tail(1)
    .sort_values("start_time")
)
casebook[[
    "small_event_id", "start_time", "target_small_mode", "prior_large_mode",
    "outcome", "resolution_time", "resolution_bars",
]]

,small_event_id,start_time,target_small_mode,prior_large_mode,outcome,resolution_time,resolution_bars
1584,S2417,2026-11-05 17:25:00+02:00,up,down,rejected,2026-11-05 23:05:00+02:00,68
1585,S2419,2026-11-06 17:10:00+02:00,up,down,propagated,2026-11-09 10:10:00+02:00,204


In [16]:
CASEBOOK_BARS_BEFORE = 120
CASEBOOK_BARS_AFTER = 120
for episode in casebook.itertuples(index=False):
    window_start_position = max(0, int(episode.start_position) - CASEBOOK_BARS_BEFORE)
    window_end_position = min(len(prices), int(episode.resolution_position) + CASEBOOK_BARS_AFTER + 1)
    case_prices = prices.iloc[window_start_position:window_end_position]
    case_fig = plot_two_threshold_directional_change(
        prices=case_prices,
        small_events=small_events,
        large_events=large_events,
        small_theta=SMALL_THETA,
        large_theta=LARGE_THETA,
        n_prices=len(case_prices),
        title=(
            f"{episode.outcome} | small DCC → {episode.target_small_mode}, "
            f"prior large mode={episode.prior_large_mode} | resolved in {episode.resolution_bars} bars"
        ),
    )
    case_fig.add_vline(x=episode.start_time, line={"color": "#111827", "dash": "dot", "width": 2})
    case_fig.add_vline(
        x=episode.resolution_time,
        line={"color": "#7c3aed", "dash": "dot", "width": 2},
    )
    case_fig.show()

## What to take from this notebook

For each recent example, answer these questions in order:

1. What were the small and large live modes immediately before the initiating small DCC?
2. Which completed small leg did that DCC confirm?
3. What changed causally at the initiating bar?
4. While the modes disagreed, did the price action look like a pullback/rebound or the beginning of a broader reversal?
5. Which later DCC resolved the disagreement?
6. At what exact bar did the realized outcome first become knowable?

The intended intuition is: **small-scale opposition is observable now; propagation is only a future outcome until the large DCC occurs.** No HMM or alpha claim should be added until these cases are visually and economically clear.

# Causal development of the live counter-scale leg

The counter move begins at the peak or trough that is only confirmed retrospectively by the initiating small DCC. At that confirmation bar we can first observe it as a live counter-scale mode. We therefore keep two clocks:

- `small_t_live`: bars from the now-confirmed extreme, including the pre-confirmation movement required to reach the DCC;
- `elapsed_bars`: bars since the initiating DCC, measuring post-confirmation development.

Live trajectories exclude the resolution bar. A rejected attempt's small live features reset on that bar, while a propagated attempt's small mode does not. Resolution is stored separately as the first time the hindsight outcome becomes known.

## Build the causal active-bar panel

While the large mode is up, its causal running extreme is the running maximum within that mode; while down, it is the running minimum. Before resolution, the reversal from this extreme divided by 0.4% is exact progress toward the large DCC. Same-bar resolutions have no unresolved post-DCC life and are audited but excluded.

In [17]:
large_mode_numeric = large_live["direction"]
large_mode_run_id = large_mode_numeric.ne(large_mode_numeric.shift()).cumsum()
large_running_high = prices.groupby(large_mode_run_id).cummax()
large_running_low = prices.groupby(large_mode_run_id).cummin()
large_running_extreme = pd.Series(
    np.where(large_mode_numeric.eq(1.0), large_running_high, large_running_low),
    index=prices.index,
    name="large_running_extreme",
).where(large_mode_numeric.notna())


from dc_basics.two_scale import (
    build_counter_scale_trajectory_panel as _build_counter_scale_trajectory_panel,
)

def build_counter_scale_trajectory_panel(episodes: pd.DataFrame, close: pd.Series,
                                         small_context: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    return _build_counter_scale_trajectory_panel(
        episodes,
        close,
        small_context,
        mode_frame=mode_frame,
        large_running_extreme=large_running_extreme,
        small_theta=SMALL_THETA,
        large_theta=LARGE_THETA,
    )


counter_leg_trajectories, trajectory_attempts = build_counter_scale_trajectory_panel(
    propagation_episodes, prices, small_live
)
trajectory_exclusion_audit = pd.Series({
    "all_resolved_attempts": len(resolved_episodes),
    "same_bar_resolutions_excluded": int(resolved_episodes["resolution_bars"].eq(0).sum()),
    "trajectory_attempts": len(trajectory_attempts),
    "active_attempt_bars": len(counter_leg_trajectories),
})
trajectory_exclusion_audit

all_resolved_attempts             1586
same_bar_resolutions_excluded       32
trajectory_attempts               1554
active_attempt_bars              98613
dtype: int64

### Information contract

Every trajectory feature is known on its row timestamp. Outcome-colored summaries are hindsight diagnostics.

In [18]:
counter_leg_information_contract = pd.DataFrame([
    ("small_tmv_live", True, "current counter move from the confirmed small extreme"),
    ("small_t_live", True, "bars since that extreme, including pre-DCC confirmation time"),
    ("small_rdc_live", True, "causal running small TMV × theta / T"),
    ("elapsed_bars", True, "bars since the initiating small DCC"),
    ("post_dcc_excursion_theta", True, "signed movement from the DCC close in small-theta units"),
    ("max_post_dcc_excursion_theta", True, "maximum favorable post-DCC excursion observed so far"),
    ("large_dcc_progress", True, "reversal from causal large running extreme divided by large theta"),
    ("future_outcome", False, "propagated/rejected; available only at resolution"),
    ("outcome_information_time", False, "timestamp at which the outcome becomes known"),
], columns=["column", "known_on_trajectory_bar", "economic_meaning"])
counter_leg_information_contract

,column,known_on_trajectory_bar,economic_meaning
0,small_tmv_live,True,current counter move from the confirmed small ...
1,small_t_live,True,"bars since that extreme, including pre-DCC con..."
2,small_rdc_live,True,causal running small TMV × theta / T
3,elapsed_bars,True,bars since the initiating small DCC
4,post_dcc_excursion_theta,True,signed movement from the DCC close in small-th...
5,max_post_dcc_excursion_theta,True,maximum favorable post-DCC excursion observed ...
6,large_dcc_progress,True,reversal from causal large running extreme div...
7,future_outcome,False,propagated/rejected; available only at resolution
8,outcome_information_time,False,timestamp at which the outcome becomes known


## Audit 5: fixed post-DCC checkpoints

At checkpoint $k$, only attempts still unresolved after $k$ bars appear. The population changes with $k$; later propagation rates and outcome-conditioned medians are diagnostic among survivors, not unconditional probabilities.

In [19]:
CHECKPOINT_BARS = [0, 5, 10, 20, 40, 80, 160]
checkpoint_rows = counter_leg_trajectories.loc[
    counter_leg_trajectories["elapsed_bars"].isin(CHECKPOINT_BARS)
].copy()
checkpoint_survival = checkpoint_rows.groupby("elapsed_bars", sort=True).agg(
    surviving_attempts=("attempt_id", "nunique"),
    later_propagation_rate=("future_outcome", lambda values: values.eq(AttemptOutcome.PROPAGATED.value).mean()),
    small_tmv_median=("small_tmv_live", "median"),
    small_t_median=("small_t_live", "median"),
    small_rdc_median=("small_rdc_live", "median"),
    max_post_dcc_excursion_median=("max_post_dcc_excursion_theta", "median"),
    large_dcc_progress_median=("large_dcc_progress", "median"),
)
checkpoint_survival["survivor_share"] = checkpoint_survival["surviving_attempts"] / len(trajectory_attempts)
checkpoint_survival.style.format({
    "surviving_attempts": "{:,}", "later_propagation_rate": "{:.2%}",
    "small_tmv_median": "{:.3f}", "small_t_median": "{:.1f}",
    "small_rdc_median": "{:.3e}", "max_post_dcc_excursion_median": "{:.3f}",
    "large_dcc_progress_median": "{:.2%}", "survivor_share": "{:.2%}",
})

,surviving_attempts,later_propagation_rate,small_tmv_median,small_t_median,small_rdc_median,max_post_dcc_excursion_median,large_dcc_progress_median,survivor_share
elapsed_bars,,,,,,,,
0,"1,554",46.53%,1.066,20.0,1.118e-04,0.000,54.26%,100.00%
5,"1,434",44.77%,1.065,26.0,7.481e-05,0.101,54.48%,92.28%
10,"1,272",44.34%,1.045,34.0,5.662e-05,0.163,54.16%,81.85%
20,"1,031",44.03%,1.069,46.0,4.109e-05,0.240,55.59%,66.34%
40,720,43.33%,1.109,70.0,2.765e-05,0.312,57.24%,46.33%
80,410,42.93%,1.112,109.0,1.882e-05,0.385,56.47%,26.38%
160,177,42.37%,1.238,189.0,1.218e-05,0.520,63.88%,11.39%


In [20]:
checkpoint_outcome_summary = checkpoint_rows.groupby(
    ["elapsed_bars", "future_outcome", "target_small_mode"], sort=True
).agg(
    surviving_attempts=("attempt_id", "size"),
    small_tmv_median=("small_tmv_live", "median"),
    small_t_median=("small_t_live", "median"),
    small_rdc_median=("small_rdc_live", "median"),
    post_dcc_excursion_median=("post_dcc_excursion_theta", "median"),
    max_post_dcc_excursion_median=("max_post_dcc_excursion_theta", "median"),
    large_dcc_progress_median=("large_dcc_progress", "median"),
)
checkpoint_outcome_summary.style.format(precision=4)

## Audit 6: survival-conditioned live trajectories

Purple/slate is the later propagated/rejected outcome; solid/dashed is new-small-up/down. Each point uses only information known by that elapsed bar, but its outcome color remains hindsight. Hover survivor counts before interpreting later horizons.

In [21]:
trajectory_fig = plot_dc_counter_scale_trajectories(counter_leg_trajectories, max_elapsed_bars=200)
trajectory_fig.show()

## Audit 7: recent live-leg casebook

The colored segment contains only unresolved bars and could have been drawn sequentially. Its color is still hindsight. The triangle is the initiating small DCC; the black cross is the separate resolution bar.

In [22]:
live_leg_casebook = (
    trajectory_attempts.sort_values("start_time")
    .groupby("future_outcome", sort=False, group_keys=False)
    .tail(1)
    .sort_values("start_time")
)
live_leg_casebook[[
    "attempt_id", "start_time", "target_small_mode", "prior_large_mode",
    "future_outcome", "resolution_time", "resolution_bars",
]]

,attempt_id,start_time,target_small_mode,prior_large_mode,future_outcome,resolution_time,resolution_bars
1584,S2417,2026-11-05 17:25:00+02:00,up,down,rejected,2026-11-05 23:05:00+02:00,68
1585,S2419,2026-11-06 17:10:00+02:00,up,down,propagated,2026-11-09 10:10:00+02:00,204


In [23]:
live_leg_casebook_fig = plot_dc_counter_scale_price_casebook(
    prices=prices, trajectories=counter_leg_trajectories, attempts=live_leg_casebook,
    bars_before=120, bars_after=120,
)
live_leg_casebook_fig.show()

## Counter-scale live-leg interpretation checklist

1. At elapsed bar zero, why is live small TMV already near or above one while post-DCC excursion is zero?
2. Can live TMV and current post-DCC excursion fall while the small mode remains unchanged?
3. Does maximum post-DCC excursion preserve information that current excursion loses during retracement?
4. How quickly does progress toward the 0.4% DCC separate later propagated and rejected paths?
5. How sharply does the survivor population shrink at later checkpoints?
6. Are apparent differences consistent for new-small-up and new-small-down?
7. Can you identify the exact bar where the outcome color first becomes knowable?

Only after these are clear should we ask whether a statistical model adds information beyond the observable live trajectory.